# Limpieza e integración del catálogo StreamView

## 1. Objetivo y alcance

Este notebook prepara las fuentes oficiales de **Movies** y **TV Shows** mediante carga, revisión estructural, limpieza, normalización, homologación, integración, validación y exportación. El resultado se guarda en `data/processed/catalogo_streamview.csv` y servirá como fuente para etapas posteriores de análisis y visualización en Looker Studio.

Quedan fuera del alcance el análisis exploratorio de negocio, los gráficos, el storytelling, los dashboards, las conclusiones estratégicas y las recomendaciones.

## 2. Configuración y carga de fuentes

Las rutas se resuelven desde la estructura del proyecto, con independencia de si el notebook se ejecuta desde la raíz o desde `notebooks/`. Los hashes SHA-256 calculados al inicio se comparan al finalizar el procesamiento para comprobar que los archivos cargados no fueron modificados durante esta ejecución; no constituyen una comparación con hashes oficiales externos.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start_path):
    # Localiza la raíz que contiene data/raw.
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró la estructura data/raw del proyecto.")


def sha256_file(file_path, chunk_size=1024 * 1024):
    # Calcula el hash SHA-256 de un archivo sin modificarlo.
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


project_root = find_project_root(Path.cwd())
raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

movies_path = raw_dir / "netflix_movies_detailed_up_to_2025.csv"
tv_shows_path = raw_dir / "netflix_tv_shows_detailed_up_to_2025.csv"
output_path = processed_dir / "catalogo_streamview.csv"

source_paths = {"Movies": movies_path, "TV Shows": tv_shows_path}
source_hashes_initial = {
    source: sha256_file(path) for source, path in source_paths.items()
}

movies = pd.read_csv(movies_path)
tv_shows = pd.read_csv(tv_shows_path)

print(f"Raíz del proyecto: {project_root}")
print(f"Movies cargadas: {len(movies):,} filas")
print(f"TV Shows cargados: {len(tv_shows):,} filas")

Raíz del proyecto: /home/tomy/Downloads/Duoc/Visualizacion/visualizacion-de-datos-StreamView-Analytics
Movies cargadas: 16,000 filas
TV Shows cargados: 16,000 filas


## 3. Revisión estructural inicial

La inspección se limita a las características necesarias para decidir la limpieza: dimensiones, columnas, tipos, nulos, diferencias de esquema y duplicados completos. No se realizan rankings ni interpretaciones de negocio.

In [2]:
structural_summary = pd.DataFrame(
    {
        "filas": [len(movies), len(tv_shows)],
        "columnas": [movies.shape[1], tv_shows.shape[1]],
        "duplicados_completos": [
            int(movies.duplicated().sum()),
            int(tv_shows.duplicated().sum()),
        ],
    },
    index=["Movies", "TV Shows"],
)

common_columns = sorted(set(movies.columns) & set(tv_shows.columns))
movies_only_columns = sorted(set(movies.columns) - set(tv_shows.columns))
tv_only_columns = sorted(set(tv_shows.columns) - set(movies.columns))

initial_dtypes = pd.concat(
    [movies.dtypes.astype(str), tv_shows.dtypes.astype(str)], axis=1
)
initial_dtypes.columns = ["Movies", "TV Shows"]

initial_nulls = pd.concat(
    [movies.isna().sum(), tv_shows.isna().sum()], axis=1
)
initial_nulls.columns = ["Movies", "TV Shows"]

display(structural_summary)
print(f"Columnas de Movies: {movies.columns.tolist()}")
print(f"Columnas de TV Shows: {tv_shows.columns.tolist()}")
print(f"Columnas comunes ({len(common_columns)}): {common_columns}")
print(f"Exclusivas de Movies: {movies_only_columns}")
print(f"Exclusivas de TV Shows: {tv_only_columns}")
display(initial_dtypes)
display(initial_nulls)

,filas,columnas,duplicados_completos
Movies,16000,18,0
TV Shows,16000,16,0


Columnas de Movies: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
Columnas de TV Shows: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average']
Columnas comunes (16): ['cast', 'country', 'date_added', 'description', 'director', 'duration', 'genres', 'language', 'popularity', 'rating', 'release_year', 'show_id', 'title', 'type', 'vote_average', 'vote_count']
Exclusivas de Movies: ['budget', 'revenue']
Exclusivas de TV Shows: []


,Movies,TV Shows
show_id,int64,int64
type,str,str
title,str,str
director,str,str
cast,str,str
country,str,str
date_added,str,str
release_year,int64,int64
rating,float64,float64
duration,float64,str


,Movies,TV Shows
show_id,0,0.0
type,0,0.0
title,0,0.0
director,132,10965.0
cast,204,1157.0
country,466,1797.0
date_added,0,0.0
release_year,0,0.0
rating,0,0.0
duration,16000,0.0


## 4. Criterios de limpieza

Los valores faltantes reales se conservan como nulos. No se crean categorías como “Desconocido” o “No disponible” y no se realiza imputación estadística.

| Variable | Tratamiento |
|---|---|
| `description` | Eliminar después de la integración. |
| `duration` | Mantener `NaN` en Movies y conservar los valores de TV Shows. |
| `country` | Mantener `NaN`; normalizar solamente el formato multivalor. |
| `genres` | Mantener `NaN`; normalizar solamente el formato multivalor, sin ordenar sus elementos. |
| `cast` | Mantener `NaN`; normalizar solamente espacios estructurales. |
| `director` | Mantener `NaN`; normalizar solamente espacios estructurales. |
| `budget` | Convertir 0 a `NaN`; aplicable únicamente a Movies. |
| `revenue` | Convertir 0 a `NaN`; aplicable únicamente a Movies. |

Solo se eliminan filas completamente idénticas. Los identificadores repetidos con información diferente no se resuelven automáticamente.

## 5. Limpieza y normalización de las fuentes

Las cadenas se recortan en sus extremos y las cadenas vacías se convierten en nulos. En las variables multivalor se uniforman únicamente los espacios alrededor de las comas, preservando el orden y el significado originales. Después se eliminan, si existen, duplicados de fila completa dentro de cada fuente.

In [3]:
MULTIVALUE_COLUMNS = ["cast", "director", "country", "genres"]


def normalize_multivalue(value):
    # Normaliza separadores sin ordenar, traducir ni recategorizar valores.
    if pd.isna(value):
        return pd.NA
    parts = [part.strip() for part in str(value).split(",") if part.strip()]
    return ", ".join(parts) if parts else pd.NA


def clean_source(dataframe):
    # Aplica únicamente transformaciones estructurales permitidas.
    cleaned = dataframe.copy()
    text_columns = cleaned.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()
        cleaned[column] = cleaned[column].replace("", pd.NA)

    for column in MULTIVALUE_COLUMNS:
        if column in cleaned.columns:
            cleaned[column] = (
                cleaned[column].map(normalize_multivalue).astype("string")
            )

    duplicate_count = int(cleaned.duplicated().sum())
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)
    return cleaned, duplicate_count


movies_clean, movie_duplicates_removed = clean_source(movies)
tv_shows_clean, tv_duplicates_removed = clean_source(tv_shows)

print(f"Duplicados completos eliminados de Movies: {movie_duplicates_removed}")
print(f"Duplicados completos eliminados de TV Shows: {tv_duplicates_removed}")
print(f"Filas limpias de Movies: {len(movies_clean):,}")
print(f"Filas limpias de TV Shows: {len(tv_shows_clean):,}")

Duplicados completos eliminados de Movies: 0
Duplicados completos eliminados de TV Shows: 0
Filas limpias de Movies: 16,000
Filas limpias de TV Shows: 16,000


## 6. Homologación de esquemas

`budget` y `revenue` son variables exclusivas de Movies. Se incorporan a TV Shows como columnas completamente nulas y luego se aplica el mismo orden de columnas a ambas fuentes antes de concatenarlas.

In [4]:
for column in ["budget", "revenue"]:
    if column not in tv_shows_clean.columns:
        tv_shows_clean[column] = pd.Series(
            pd.NA, index=tv_shows_clean.index, dtype="Float64"
        )

column_order = movies_clean.columns.tolist()
tv_shows_clean = tv_shows_clean.reindex(columns=column_order)

assert movies_clean.columns.tolist() == tv_shows_clean.columns.tolist()
assert tv_shows_clean[["budget", "revenue"]].isna().all().all()

print(f"Esquemas homologados: {len(column_order)} columnas por fuente")

Esquemas homologados: 18 columnas por fuente


## 7. Integración

Movies y TV Shows se integran mediante concatenación vertical. No se emplean `join` ni `merge`, por lo que se mantiene la granularidad de las fuentes: cada registro de entrada aporta una fila al catálogo. La columna `description` se elimina después de la unión, conforme a los criterios definidos.

In [5]:
catalogo = pd.concat([movies_clean, tv_shows_clean], ignore_index=True)
catalogo = catalogo.drop(columns="description")

print(f"Catálogo integrado: {catalogo.shape[0]:,} filas y {catalogo.shape[1]} columnas")

Catálogo integrado: 32,000 filas y 17 columnas


## 8. Transformaciones finales

Los tipos se normalizan sobre el catálogo integrado. La conversión de `date_added` conserva los nulos preexistentes y se detiene si aparecen nulos nuevos. En `budget` y `revenue`, los ceros se interpretan como información financiera no disponible según la regla del proyecto y se convierten en nulos.

In [6]:
text_columns = [
    "show_id", "type", "title", "director", "cast", "country",
    "duration", "genres", "language",
]
integer_columns = ["release_year", "vote_count"]
float_columns = ["rating", "popularity", "vote_average"]
financial_columns = ["budget", "revenue"]

for column in text_columns:
    catalogo[column] = catalogo[column].astype("string")

date_nulls_before = int(catalogo["date_added"].isna().sum())
converted_dates = pd.to_datetime(
    catalogo["date_added"], format="%Y-%m-%d", errors="coerce"
)
date_nulls_after = int(converted_dates.isna().sum())
unexpected_date_nulls = date_nulls_after - date_nulls_before
assert unexpected_date_nulls == 0, (
    f"La conversión de date_added introdujo {unexpected_date_nulls} nulos."
)
catalogo["date_added"] = converted_dates

for column in integer_columns:
    catalogo[column] = pd.to_numeric(
        catalogo[column], errors="raise"
    ).astype("Int64")

for column in float_columns:
    catalogo[column] = pd.to_numeric(
        catalogo[column], errors="raise"
    ).astype("Float64")

for column in financial_columns:
    catalogo[column] = pd.to_numeric(
        catalogo[column], errors="raise"
    ).astype("Float64")
    catalogo[column] = catalogo[column].replace(0, pd.NA)

print(f"Nulos nuevos por conversión de date_added: {unexpected_date_nulls}")

Nulos nuevos por conversión de date_added: 0


## 9. Validación integral

Las comprobaciones siguientes centralizan las reglas de volumen, esquema, granularidad, tipos, nulos y restricciones financieras. La búsqueda de representaciones artificiales de ausencia compara el valor completo después de aplicar `strip()` y `casefold()`; no utiliza búsquedas por subcadena, por lo que valores válidos como `Sci-Fi` no producen falsos positivos.

In [7]:
expected_columns = [
    "show_id", "type", "title", "director", "cast", "country",
    "date_added", "release_year", "rating", "duration", "genres",
    "language", "popularity", "vote_count", "vote_average",
    "budget", "revenue",
]
expected_type_counts = {"Movie": 16_000, "TV Show": 16_000}
artificial_na_tokens = {
    "n/a", "no disponible", "desconocido", "null", "none", "-"
}
artificial_na_columns = ["duration", "genres", "cast", "director", "country"]

artificial_na_counts = {}
for column in artificial_na_columns:
    normalized_values = catalogo[column].dropna().str.strip().str.casefold()
    artificial_na_counts[column] = int(
        normalized_values.isin(artificial_na_tokens).sum()
    )

type_counts = catalogo["type"].value_counts().to_dict()
full_duplicates = int(catalogo.duplicated().sum())
repeated_show_id_counts = catalogo["show_id"].value_counts()
repeated_show_ids = repeated_show_id_counts[repeated_show_id_counts.gt(1)]
source_hashes_current = {
    source: sha256_file(path) for source, path in source_paths.items()
}

assert catalogo.shape == (32_000, 17)
assert catalogo.columns.tolist() == expected_columns
assert "description" not in catalogo.columns
assert type_counts == expected_type_counts
assert set(catalogo["type"].dropna().unique()) == {"Movie", "TV Show"}
assert full_duplicates == 0
assert int(catalogo["budget"].eq(0).sum()) == 0
assert int(catalogo["revenue"].eq(0).sum()) == 0
assert int(catalogo["budget"].isna().sum()) == 27_153
assert int(catalogo["revenue"].isna().sum()) == 26_355
assert catalogo.loc[
    catalogo["type"].eq("TV Show"), ["budget", "revenue"]
].isna().all().all()
assert str(catalogo["budget"].dtype) == "Float64"
assert str(catalogo["revenue"].dtype) == "Float64"
assert sum(artificial_na_counts.values()) == 0
assert source_hashes_current == source_hashes_initial

validation_summary = pd.Series(
    {
        "filas": len(catalogo),
        "columnas": catalogo.shape[1],
        "Movies": type_counts.get("Movie", 0),
        "TV Shows": type_counts.get("TV Show", 0),
        "duplicados completos": full_duplicates,
        "show_id repetidos": len(repeated_show_ids),
        "filas con show_id repetido": int(repeated_show_ids.sum()),
        "ceros en budget": int(catalogo["budget"].eq(0).sum()),
        "ceros en revenue": int(catalogo["revenue"].eq(0).sum()),
        "nulos en budget": int(catalogo["budget"].isna().sum()),
        "nulos en revenue": int(catalogo["revenue"].isna().sum()),
        "tokens artificiales de nulo": sum(artificial_na_counts.values()),
    },
    name="resultado",
)

display(validation_summary.to_frame())
display(catalogo.isna().sum().rename("nulos").to_frame())
display(catalogo.dtypes.astype(str).rename("tipo").to_frame())
display(pd.Series(artificial_na_counts, name="coincidencias exactas").to_frame())

,resultado
filas,32000
columnas,17
Movies,16000
TV Shows,16000
duplicados completos,0
show_id repetidos,406
filas con show_id repetido,812
ceros en budget,0
ceros en revenue,0
nulos en budget,27153


,nulos
show_id,0
type,0
title,0
director,11097
cast,1361
country,2263
date_added,0
release_year,0
rating,0
duration,16000


,tipo
show_id,string
type,string
title,string
director,string
cast,string
country,string
date_added,datetime64[us]
release_year,Int64
rating,Float64
duration,string


,coincidencias exactas
duration,0
genres,0
cast,0
director,0
country,0


Los `show_id` repetidos se conservan porque no corresponden a filas completamente idénticas y no existe evidencia suficiente para eliminarlos automáticamente.

## 10. Exportación

Superadas las validaciones, `date_added` se representa como `YYYY-MM-DD` y el catálogo se exporta una sola vez al directorio de datos procesados. No se generan versiones intermedias del dataset definitivo.

In [8]:
processed_dir.mkdir(parents=True, exist_ok=True)

catalogo_export = catalogo.copy()
catalogo_export["date_added"] = catalogo_export["date_added"].dt.strftime(
    "%Y-%m-%d"
)
catalogo_export.to_csv(output_path, index=False)

print(f"Archivo exportado: {output_path}")

Archivo exportado: /home/tomy/Downloads/Duoc/Visualizacion/visualizacion-de-datos-StreamView-Analytics/data/processed/catalogo_streamview.csv


## 11. Verificación posterior a la exportación

El único archivo definitivo se vuelve a cargar con los tipos esperados. Se comprueban nuevamente dimensiones, columnas, tipos de contenido, duplicados completos, recuperación de fecha y numéricos, y las restricciones de `budget` y `revenue`.

In [9]:
export_dtypes = {
    **{column: "string" for column in text_columns},
    **{column: "Int64" for column in integer_columns},
    **{column: "Float64" for column in float_columns + financial_columns},
}
catalogo_verified = pd.read_csv(output_path, dtype=export_dtypes)
catalogo_verified["date_added"] = pd.to_datetime(
    catalogo_verified["date_added"], format="%Y-%m-%d", errors="raise"
)

assert catalogo_verified.shape == catalogo.shape
assert catalogo_verified.columns.tolist() == expected_columns
assert catalogo_verified["type"].value_counts().to_dict() == expected_type_counts
assert set(catalogo_verified["type"].dropna().unique()) == {"Movie", "TV Show"}
assert int(catalogo_verified.duplicated().sum()) == 0
assert catalogo_verified["date_added"].isna().sum() == catalogo["date_added"].isna().sum()
assert all(
    catalogo_verified[column].dtype == catalogo[column].dtype
    for column in integer_columns + float_columns + financial_columns
)
assert int(catalogo_verified["budget"].eq(0).sum()) == 0
assert int(catalogo_verified["revenue"].eq(0).sum()) == 0
assert int(catalogo_verified["budget"].isna().sum()) == 27_153
assert int(catalogo_verified["revenue"].isna().sum()) == 26_355
assert catalogo_verified.loc[
    catalogo_verified["type"].eq("TV Show"), ["budget", "revenue"]
].isna().all().all()

print(
    "Verificación posterior a la exportación superada: "
    f"{catalogo_verified.shape[0]:,} filas y {catalogo_verified.shape[1]} columnas."
)

Verificación posterior a la exportación superada: 32,000 filas y 17 columnas.


## Cierre

Las fuentes de Movies y TV Shows fueron preparadas e integradas de acuerdo con las reglas del proyecto. El catálogo resultante fue validado antes y después de su exportación, y `data/processed/catalogo_streamview.csv` queda preparado para las etapas posteriores de análisis y visualización en Looker Studio.